# `BisectionChemicalEquilibriumEngine` — Instantiation and Use

Three engines satisfy `ChemicalEquilibriumEngineProtocol`:
`BisectionChemicalEquilibriumEngine`, `NRChemicalEquilibriumEngine`, and
`PHREEQCChemicalEquilibriumEngine`. None is "the" default — pick the one
that matches your chemistry's shape. This notebook covers
`BisectionChemicalEquilibriumEngine`: PyOMES's original speciation engine, a
1-D bisection solver over the charge balance, scanning `pH_min`–`pH_max`
for the root. It solves single acid-base ladders (and independent ladders
that only interact through the charge balance, e.g. carbonate + ammonia)
declared as `EquilibriumReaction` instances.

**Contrast with `NRChemicalEquilibriumEngine`:** the NR engine builds a
log-linear tableau and solves the full multi-variable Newton system, so it
handles arbitrary cross-component networks (species whose mass balance
couples to more than one "total") and gas-liquid/solid-liquid folding.
`BisectionChemicalEquilibriumEngine` cannot do either of those — it is the right
choice when your chemistry is a small number of decoupled acid-base ladders
and you want the simpler, historically-validated bisection path. See
[`02_nr_engine_basics.ipynb`](02_nr_engine_basics.ipynb) for the NR engine's
own instantiation/call conventions, including the two capabilities this
engine structurally cannot have, or
[`03_phreeqc_engine_basics.ipynb`](03_phreeqc_engine_basics.ipynb) for the
PHREEQC-backed engine, which forgoes declared `EquilibriumReaction`
networks entirely in favour of PHREEQC's own database.

This notebook focuses on the mechanics of the engine itself: constructing
it, calling `solve()`, and reading back an `EquilibriumResult`. For
engine-to-engine accuracy comparisons against `NRChemicalEquilibriumEngine`,
see
[`../../model_api/chemistry/speciation/02_multi_component_systems.ipynb`](../../model_api/chemistry/speciation/02_multi_component_systems.ipynb)
and
[`../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb`](../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb).

In [1]:
import sys
from pathlib import Path

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    CO2, HCO3_minus, CO3_2minus,
    NH3, NH4_plus,
)
from PyOMES.reactions.equilibrium import EquilibriumReaction
from PyOMES.reactions.stoichiometry import StoichiometryEntry
from PyOMES.chemical_equilibrium.engine import BisectionChemicalEquilibriumEngine

def _e(sp, coeff, phase="liquid"):
    return StoichiometryEntry(species=sp, phase=phase, coefficient=coeff)

print("Imports OK")


Imports OK


## 1  Instantiation via `from_reactions()`

`BisectionChemicalEquilibriumEngine.from_reactions()` classifies each declared
`EquilibriumReaction` into `"water"`, `"acid"`, or `"cation_acid"` from its
stoichiometry, and folds them into an internal `EquilibriumSet`
(`engine._equilibrium_set`). Reactions that share a `total_id` (e.g. a
polyprotic ladder) are merged into a single `EquilibriumDef` automatically.

Any item that is *not* a single-phase acid-base equilibrium (a gas-liquid
`HenryEquilibrium`, a solid-liquid `KspEquilibrium`, etc.) is **not silently
dropped** — it is diverted to `engine.cross_phase_constraints` instead of
being folded into the tableau. Section 5 below shows why that matters.

In [2]:
water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
co2_first = EquilibriumReaction(
    stoichiometry=[_e(CO2, -1), _e(H2O, -1), _e(HCO3_minus, +1), _e(H_plus, +1)],
    log_K=-6.35, total_id="CO2", label="co2_first",
)
co2_second = EquilibriumReaction(
    stoichiometry=[_e(HCO3_minus, -1), _e(CO3_2minus, +1), _e(H_plus, +1)],
    log_K=-10.33, total_id="CO2", label="co2_second",
)

engine = BisectionChemicalEquilibriumEngine.from_reactions([water, co2_first, co2_second])

print("Engine constructed:  ", type(engine).__name__)
print("EquilibriumSet names:", [d.name for d in engine._equilibrium_set])
print("Cross-phase constraints (empty — no gas/solid items declared):",
      engine.cross_phase_constraints)


Engine constructed:   BisectionChemicalEquilibriumEngine
EquilibriumSet names: ['CO2']
Cross-phase constraints (empty — no gas/solid items declared): ()


## 2  `solve()` and the total-key mapping

Unlike `NRChemicalEquilibriumEngine.solve(totals={...})`,
`BisectionChemicalEquilibriumEngine.solve()` takes totals as **named keyword
arguments**, using a fixed mapping from species id to a conventional total
key (`_TOTAL_KEY_OVERRIDES` in `engine.py`):

| Species id | Total kwarg |
|---|---|
| `CO2` | `CT_TIC` |
| `NH3` / `NH4+` | `CT_NH_T` |
| `H3PO4` | `CT_P` |
| `SO4--` | `CT_SO4` |

`solve()` returns an immutable `EquilibriumResult` — the same return type
`NRChemicalEquilibriumEngine` and `PHREEQCChemicalEquilibriumEngine` use.
`species_mol_L` is restricted to a fixed canonical tuple this engine has
always written back (`_CANONICAL_WRITEBACK_SPECIES`); anything else lands in
`.extra`.

**Field coverage differs by engine.** `charge_residual` is populated by the
NR solver (it's the last row of the Newton residual) but `solve_acid_base`
and `solve_from_equilibrium_set` — the two paths behind
`BisectionChemicalEquilibriumEngine.solve()` — never set it, so it's always `None`
here. `pH`, `species_mol_L`, and `ionic_strength` are populated on every
engine; don't assume every `EquilibriumResult` field is.

In [3]:
result = engine.solve(CT_TIC=0.01)   # 10 mmol/L total inorganic carbon

print(f"pH               = {result.pH:.4f}")
print(f"species_mol_L    = {result.species_mol_L}")
print(f"charge_residual  = {result.charge_residual!r}  (always None for this engine)")
print(f"ionic_strength   = {result.ionic_strength:.4e} mol/L")
print(f"alphas           = {result.alphas}")


pH               = 4.1765
species_mol_L    = {'H+': 6.661154521892359e-05, 'OH-': 1.5012412588740116e-10, 'CO2': 0.009933388651678546, 'HCO3-': 6.661130154811157e-05, 'CO3--': 4.677334302714678e-11, 'K+': 0.0, 'Na+': 0.0, 'Cl-': 0.0, 'Mg++': 0.0, 'Ca++': 0.0, 'Zn++': 0.0, 'Mn++': 0.0, 'Co++': 0.0, 'Mo7O24------': 0.0}
charge_residual  = None  (always None for this engine)
ionic_strength   = 6.6612e-05 mol/L
alphas           = {'CO2aq': 0.9933388651678545}


### 2.1  Gotcha: NR-style `totals=` is silently ignored

`solve()` accepts `**kwargs` and only ever looks for the specific total
keywords above — it never reads a `totals=` dict. Passing NR-style
`totals={"CO2": 0.01}` compiles and runs, but every total defaults to `0.0`
because `CT_TIC` was never set. There is no error; the result is just wrong.
This is exactly the kind of engine-convention mismatch this notebook exists
to make obvious.

In [4]:
result_wrong = engine.solve(totals={"CO2": 0.01})   # WRONG for this engine
result_right = engine.solve(CT_TIC=0.01)             # correct

print(f"pH with totals={{'CO2': 0.01}} (ignored): {result_wrong.pH:.4f}  ← defaults to CT_TIC=0")
print(f"pH with CT_TIC=0.01 (correct):           {result_right.pH:.4f}")


pH with totals={'CO2': 0.01} (ignored): 7.0000  ← defaults to CT_TIC=0
pH with CT_TIC=0.01 (correct):           4.1765


## 3  Strong ions and the Davies activity model

Strong ions are also passed as direct keyword arguments (`CT_Na`, `CT_Cl`,
`CT_Ca`, ... — see the list in `engine.py`), not a nested `strong_ions=`
dict. Activity corrections are configured once at construction time via
`use_activity=` / `activity_model=`, exactly as for `NRChemicalEquilibriumEngine`.

In [5]:
baseline = engine.solve(CT_TIC=0.01)
dosed    = engine.solve(CT_TIC=0.01, CT_Na=0.01)

print(f"pH baseline (no strong ions):  {baseline.pH:.4f}")
print(f"pH with 10 mmol/L Na+ dosing:  {dosed.pH:.4f}")

# Like-for-like ideal-vs-Davies comparison: same totals AND same strong-ion
# composition (10 mmol/L NaCl — net-neutral charge), only the activity
# model differs between the two engines.
engine_davies = BisectionChemicalEquilibriumEngine.from_reactions(
    [water, co2_first, co2_second], use_activity=True, activity_model="davies",
)
ideal_nacl  = engine.solve(CT_TIC=0.01, CT_Na=0.01, CT_Cl=0.01)
davies_nacl = engine_davies.solve(CT_TIC=0.01, CT_Na=0.01, CT_Cl=0.01)

print(f"\npH ideal (γ=1),   +10 mmol/L NaCl: {ideal_nacl.pH:.4f}")
print(f"pH Davies-corrected, +10 mmol/L NaCl: {davies_nacl.pH:.4f}")
print(f"Ionic strength (Davies run):          {davies_nacl.ionic_strength*1e3:.2f} mmol/L")


pH baseline (no strong ions):  4.1765
pH with 10 mmol/L Na+ dosing:  8.3353

pH ideal (γ=1),   +10 mmol/L NaCl: 4.1765
pH Davies-corrected, +10 mmol/L NaCl: 4.2215
Ionic strength (Davies run):          10.07 mmol/L


## 4  Multi-component: independent acid-base ladders

`from_reactions()` can bind more than one acid family at once — each
`total_id` group becomes its own `EquilibriumDef`, and the bisection solves
the *combined* charge balance across all of them simultaneously. This is
different from cross-component *coupling* (e.g. a species whose mass balance
spans two totals, which this engine cannot represent): here, carbonate and
ammonia only interact through the shared pH.

In [6]:
nh4 = EquilibriumReaction(
    stoichiometry=[_e(NH4_plus, -1), _e(NH3, +1), _e(H_plus, +1)],
    log_K=-9.25, total_id="NH3", label="nh4",
)

engine_multi = BisectionChemicalEquilibriumEngine.from_reactions([water, co2_first, co2_second, nh4])
result_multi = engine_multi.solve(CT_TIC=0.01, CT_NH_T=0.04)

print("EquilibriumSet names:", [d.name for d in engine_multi._equilibrium_set])
print(f"pH                  = {result_multi.pH:.4f}")
print(f"species_mol_L       = {result_multi.species_mol_L}")
print()
print("For a full NR-vs-charge-balance accuracy sweep on this exact carbonate")
print("+ ammonia system, see 02_multi_component_systems.ipynb in the")
print("model_api/chemistry/speciation/ demo family.")


EquilibriumSet names: ['CO2', 'NH3']
pH                  = 9.6330
species_mol_L       = {'H+': 2.3279513240179008e-10, 'OH-': 4.295622462904686e-05, 'CO2': 4.337813708146613e-06, 'HCO3-': 0.008323327851599534, 'CO3--': 0.001672334334692319, 'NH4+': 0.011710952512817669, 'NH3': 0.028289047487182334, 'K+': 0.0, 'Na+': 0.0, 'Cl-': 0.0, 'Mg++': 0.0, 'Ca++': 0.0, 'Zn++': 0.0, 'Mn++': 0.0, 'Co++': 0.0, 'Mo7O24------': 0.0}

For a full NR-vs-charge-balance accuracy sweep on this exact carbonate
+ ammonia system, see 02_multi_component_systems.ipynb in the
model_api/chemistry/speciation/ demo family.


## 5  Gotcha: gas-liquid/solid-liquid items are diverted, not solved

If a `HenryEquilibrium` (or `KspEquilibrium`/`RaoultEquilibrium`) is included
in the reaction list, `from_reactions()` classifies it as `"gas_liquid"` (or
`"solid_liquid"`) and routes it to `engine.cross_phase_constraints` instead
of folding it into the acid-base tableau. `solve()` never reads
`cross_phase_constraints` — the item is exposed for a caller such as
`KineticGasLiquidLink` to pick up, not solved by this engine at all.

Declaring a Henry term and expecting it to affect the liquid-phase pH here
is a real mistake this API shape invites; the item is preserved (not
discarded) precisely so a caller can detect it and route it elsewhere —
but `BisectionChemicalEquilibriumEngine.solve()` itself will not fold it in. Use
`NRChemicalEquilibriumEngine` (see `LAYER1_GAP_CLOSURE`'s gas-liquid folding)
when you need the Henry/Raoult term to enter the same solve.

In [7]:
from PyOMES.chemistry import HenryEquilibrium

co2_henry = HenryEquilibrium(H_ref=3.4e-4, dlnH=2400.0, gas_species="CO2", liquid_species="CO2",
                              label="co2_henry")

engine_mixed = BisectionChemicalEquilibriumEngine.from_reactions([water, co2_first, co2_second, co2_henry])

print("Cross-phase constraints found:", [c.label for c in engine_mixed.cross_phase_constraints])

result_mixed = engine_mixed.solve(CT_TIC=0.01)
print(f"pH with Henry term declared:  {result_mixed.pH:.4f}")
print(f"pH without it (Section 2):    {result.pH:.4f}")
print("(identical — the Henry term never entered the charge balance)")


Cross-phase constraints found: ['co2_henry']
pH with Henry term declared:  4.1765
pH without it (Section 2):    4.1765
(identical — the Henry term never entered the charge balance)


## 6  Convenience helpers

- `get_CO2aq_from_totals(**kwargs)` — one-shot CO2(aq) lookup, checking both
  `species_mol_L` and `extra` since "CO2aq" is not in the canonical
  writeback tuple.
- `logH_warmstart` / `I_warmstart` — the current warm-start cache (used to
  seed the next bisection scan when `use_warmstart=True`, the default).
- `reset_cache()` / `reset_counters()` — clear the warm-start cache / the
  `n_solve_calls` counter.
- `algebraic_species()` — species ids written to `phase.n_mol` by
  `_refresh_derived`; the algebraic state `z` in a DAE formulation.

In [8]:
print("get_CO2aq_from_totals:", engine.get_CO2aq_from_totals(CT_TIC=0.01))
print("logH_warmstart:        ", engine.logH_warmstart)
print("I_warmstart:           ", engine.I_warmstart)
print("n_solve_calls so far:  ", engine.n_solve_calls)

engine.reset_cache()
engine.reset_counters()
print("\nafter reset_cache()/reset_counters():")
print("logH_warmstart:        ", engine.logH_warmstart)
print("n_solve_calls:         ", engine.n_solve_calls)

print("\nalgebraic_species():  ", sorted(engine.algebraic_species()))


get_CO2aq_from_totals: 0.00993338865167858
logH_warmstart:         -4.176450491695454
I_warmstart:            6.661159199226668e-05
n_solve_calls so far:   7

after reset_cache()/reset_counters():
logH_warmstart:         None
n_solve_calls:          0

algebraic_species():   ['CO2', 'CO3--', 'H+', 'HCO3-', 'OH-']


## Summary

| Topic | Key takeaway |
|---|---|
| Construction | `from_reactions()` classifies each item; non-acid-base items go to `cross_phase_constraints`, not the tableau |
| `solve()` totals | Fixed keyword mapping (`CT_TIC`, `CT_NH_T`, `CT_P`, `CT_SO4`) — **not** `totals={...}` |
| Strong ions | Direct kwargs (`CT_Na`, `CT_Ca`, ...), not a nested dict |
| Result type | `EquilibriumResult`, same as the NR and PHREEQC engines; `species_mol_L` restricted to a fixed canonical tuple |
| Multi-component | Independent acid ladders solve together via the shared charge balance; true cross-component coupling needs `NRChemicalEquilibriumEngine` |
| Gas-liquid items | Diverted to `cross_phase_constraints`, never folded into `solve()` — a common source of silent no-ops |
| Helpers | `get_CO2aq_from_totals`, warm-start cache properties, `reset_cache`/`reset_counters`, `algebraic_species()` |